# Pokrytí mamografickým screeningem: revize závěrů

Doprovodný notebook k explorační analýze `mamograf_exploration.ipynb`. Bere její
hlavní zjištění a podrobuje je přísnějším testům:

| Kap. | Otázka | Proč |
|---|---|---|
| 1 | Není rozdíl mezi okresy jen věková skladba? | pokrytí 45+ je hrubé, po 75 letech prudce klesá |
| 2 | Opravdu leží pětina rozdílů mezi kraji? | se 14 kraji a 77 okresy vyjde zhruba 17 % i čistou náhodou |
| 3 | Nesouvisí pokrytí s dostupností, když ji změříme lépe? | dosud jen vzdálenost k nejbližšímu pracovišti |
| 4 | Co vysvětluje pokrytí mimo vzorek? | model s 22 prediktory na 77 okresů je přeučený a míchá příčiny s důsledky |
| 5 | Jsou okresy nezávislé pozorování? | sousední okresy si jsou podobné, *p* hodnoty pak vycházejí příliš optimisticky |
| 6 | Jdou spolu i *změny* v čase? | panel 2010–2024 explorace využila jen pro animaci |
| 7 | Co z původních závěrů platí | souhrnná tabulka |

Data jsou stejná jako v exploraci, načítá je `utils/nacteni.py` (jen z lokální
cache v `data/`, nic se nestahuje). Analýza je pořád **ekologická**: jednotkou je
okres, ne žena.

In [1]:
import sys
import warnings
from pathlib import Path

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "nacteni.py").exists())
sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.linear_model import LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from utils import analyza as an
from utils import nacteni as nac
from utils.vzhled import (AXIS, BLUES, DIV, GRID, INK2, MUTED, S1, S2, SURFACE,
                          cz, layout, mapa_geo, skala)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", module="statsmodels")
pd.set_option("display.max_columns", 40)
pio.renderers.default = "plotly_mimetype+notebook_connected"

OUT = BASE / "outputs" / "analyza"
OUT.mkdir(parents=True, exist_ok=True)
ROK, TOTAL, BANDS, POPIS = nac.ROK, nac.TOTAL, nac.BANDS, nac.POPIS

V = nac.vse()
D, X, GEO, CENTROID, centra, nr, toky, OKRES = (
    V[k] for k in ("D", "X", "GEO", "CENTROID", "centra", "nr", "toky", "OKRES"))
LAUS = sorted(CENTROID)
X = X.set_index("lau").reindex(LAUS)
RNG = np.random.default_rng(2024)
print(f"{len(LAUS)} okresů, panel {D.rok.min()}–{D.rok.max()}, {len(centra)} pracovišť")

77 okresů, panel 2010–2024, 75 pracovišť


## 0 · Kontrola shody s explorací

Načítání je přepsané z explorace do modulu. Než na něm cokoli postavíme,
ověříme, že dává stejná čísla.

In [2]:
t = D[(D.rok == ROK) & (D.band == TOTAL)]
cr = 100 * (t.pocet_vysetreni.sum() + t.mamo_predchozi.sum()) / t.zeny.sum()
o = t.set_index("okres").mamo_2y.round(1)
r_gyn = stats.pearsonr(X.gyn_prevence_pct, X.mamo_2y)[0]
assert round(cr, 1) == 55.4, cr
assert o["Třebíč"] == 67.0 and o["Jeseník"] == 39.8
assert round(r_gyn, 3) == 0.463, r_gyn
print(f"ČR {ROK}: {cr:.1f} %, Třebíč {o['Třebíč']} %, Jeseník {o['Jeseník']} %, "
      f"r(gynekologie, mamograf) = {r_gyn:.3f}  ✓ shoda s explorací")

ČR 2024: 55.4 %, Třebíč 67.0 %, Jeseník 39.8 %, r(gynekologie, mamograf) = 0.463  ✓ shoda s explorací


## 1 · Věkově standardizované pokrytí

Hrubé pokrytí 45+ je vážený průměr pokrytí v pásmech, s váhami podle věkové
skladby okresu. Pokrytí je do 74 let kolem 60 % a pak prudce klesá, takže okres
se starší populací by měl vycházet hůř, i kdyby se v každém pásmu choval stejně.

**Přímá standardizace:** pokrytí v každém pásmu převážíme jednotnou věkovou
skladbou žen 45+ celé ČR (průměr 2020–2024). Pokrytí počítáme z počtů sečtených
za 2020–2024 (dvouleté okno, stejně jako v exploraci). Kromě toho počítáme pokrytí
jádra cílové skupiny **50–69 let**, na kterou míří evropská doporučení.

In [3]:
P = an.pasma(D)                    # pokrytí po okresech a pásmech, 2020–2024
STANDARD = an.standard(P)          # věková skladba žen 45+ celé ČR
JADRO = an.JADRO

def standardizuj(col, pasma=BANDS):
    return an.standardizuj(P, col, STANDARD, pasma)

S = pd.DataFrame({
    "m_hruby": P.groupby("lau").mamo.sum() / P.groupby("lau").zeny.sum() * 100,
    "m_std":   standardizuj("m"),
    "m_5069":  standardizuj("m", JADRO),
    "p_std":   standardizuj("p"),
    "podil_75": 100 * (P[P.band.isin(["75–79", "80–84", "85+"])].groupby("lau").zeny.sum()
                       / P.groupby("lau").zeny.sum()),
}).reindex(LAUS)
S["okres"], S["kraj"] = X.okres, X.kraj

# kontrola: při standardu = populace ČR musí národní standardizované pokrytí
# vyjít stejně jako hrubé
narod = P.groupby("band")[["mamo", "zeny"]].sum()
assert np.isclose((STANDARD * 100 * narod.mamo / narod.zeny).sum(),
                  100 * narod.mamo.sum() / narod.zeny.sum())

STD_POPIS = {"m_hruby": "hrubé pokrytí 45+", "m_std": "standardizované 45+",
             "m_5069": "50–69 let", "p_std": "prevence u PL, standardizovaná"}
rows = []
for c in ["m_hruby", "m_std", "m_5069"]:
    rows.append({"pokrytí": STD_POPIS[c], "průměr okresů %": S[c].mean(),
                 "SD okresů p. b.": S[c].std(),
                 "r s podílem žen 75+": stats.pearsonr(S[c], S.podil_75)[0],
                 "r s průměrným věkem žen": stats.pearsonr(S[c], X.prumerny_vek_zeny)[0]})
VEK = pd.DataFrame(rows).set_index("pokrytí")
posun = (S.m_hruby.rank(ascending=False) - S.m_std.rank(ascending=False))
print("standardní věková skladba:", ", ".join(f"{b} {100*w:.1f} %" for b, w in STANDARD.items()))
print(f"posun v pořadí okresů po standardizaci: průměr {posun.abs().mean():.1f}, "
      f"max {posun.abs().max():.0f} míst ({S.okres[posun.abs().idxmax()]})")
VEK.round(3)

standardní věková skladba: 45–49 16.3 %, 50–54 13.3 %, 55–59 12.4 %, 60–64 11.5 %, 65–69 12.8 %, 70–74 12.6 %, 75–79 9.9 %, 80–84 6.1 %, 85+ 5.2 %
posun v pořadí okresů po standardizaci: průměr 2.5, max 11 míst (Praha-západ)


,průměr okresů %,SD okresů p. b.,r s podílem žen 75+,r s průměrným věkem žen
pokrytí,,,,
hrubé pokrytí 45+,53.431,4.447,0.149,-0.047
standardizované 45+,53.330,4.578,0.262,0.021
50–69 let,60.439,4.738,0.265,0.044


In [4]:
sc = go.Figure()
sc.add_trace(go.Scatter(x=[S.m_hruby.min() - 1, S.m_hruby.max() + 1],
                        y=[S.m_hruby.min() - 1, S.m_hruby.max() + 1], mode="lines",
                        line=dict(color=AXIS, dash="dot"), hoverinfo="skip",
                        name="beze změny"))
sc.add_trace(go.Scatter(
    x=S.m_hruby, y=S.m_std, mode="markers", name="okres",
    marker=dict(size=9, color=S.podil_75, colorscale=BLUES, line=dict(color=SURFACE, width=1.5),
                colorbar=skala("ženy 75+<br>mezi 45+")),
    customdata=np.c_[S.okres, S.podil_75.round(1), posun],
    hovertemplate=("<b>%{customdata[0]}</b><br>hrubé %{x:.1f} %<br>standardizované %{y:.1f} %"
                   "<br>žen 75+: %{customdata[1]} %<br>posun v pořadí %{customdata[2]:+.0f}"
                   "<extra></extra>")))
sc.update_layout(**layout(
    "Věková standardizace pořadí okresů skoro nemění",
    f"pokrytí mamografií za 2 roky, 2020–2024 · průměrný posun v pořadí "
    f"{posun.abs().mean():.1f} místa · barva = podíl žen 75+"))
sc.update_layout(height=520, xaxis_title="hrubé pokrytí 45+ [%]",
                 yaxis_title="věkově standardizované pokrytí 45+ [%]", showlegend=False,
                 margin=dict(l=60, r=30, t=80, b=60))
sc.write_html(OUT / "standardizace.html", include_plotlyjs="cdn")
sc

Proč tedy v modelu explorace vycházel **průměrný věk žen** jako nejsilnější
prediktor (β = −2,4 p. b.)? Sám o sobě s pokrytím nesouvisí. Záporný je jen
*podmíněně*, po odečtení 21 dalších prediktorů, se kterými je silně svázaný.

In [5]:
OLD = ["vs_zeny_pct", "nezam_zeny_pct", "prisp_bydleni_Kc_obyv", "invalid_duchod_zeny_pct",
       "prumerny_vek_zeny", "cizinky_pct", "vdovy_rozvedene_pct", "obce_0_pct", "obce_4_pct",
       "volby_ANO_pct", "dojizdka_mamo_pct", "gyn_prevence_pct",
       "diag_prs_na100", "nezhoubne_prs_na1000", "rakovina_prs_na1000",
       "cytologie_pct", "toks_pct", "diabetes_pct", "hypertenze_pct",
       "chopn_pct", "deprese_pct", "obezita_pct"]          # model z kap. 8.4 explorace

def vif(cols, df=X):
    return an.vif(df, cols)

VIF_OLD = vif(OLD)
r2_vek = 1 - 1 / VIF_OLD["prumerny_vek_zeny"]
print(f"průměrný věk žen × pokrytí:   r = {stats.pearsonr(X.prumerny_vek_zeny, X.mamo_2y)[0]:+.3f}")
print(f"průměrný věk z ostatních 21 prediktorů: R² = {r2_vek:.2f} (VIF {VIF_OLD['prumerny_vek_zeny']:.1f})")
print("\nnejvyšší VIF ve starém modelu:")
print(VIF_OLD.sort_values(ascending=False).head(6).rename(lambda c: POPIS[c][0]).round(1).to_string())

průměrný věk žen × pokrytí:   r = -0.044
průměrný věk z ostatních 21 prediktorů: R² = 0.78 (VIF 4.6)

nejvyšší VIF ve starém modelu:
vdovy a rozvedené                    8.3
vysokoškolačky, ženy 15+             7.6
příspěvek na bydlení na obyvatele    6.9
volby 2025: ANO                      6.4
podíl nezaměstnaných žen             4.9
průměrný věk žen                     4.6


**Jak to číst.**

* Předpoklad, že rozdíly mezi okresy dělá věková skladba, **se nepotvrdil**.
  Standardizace posune okresy v pořadí v průměru o 2–3 místa. Hrubé pokrytí s podílem
  nejstarších žen dokonce slabě *kladně* souvisí. Okresy se staršími ženami nejsou
  pozadu, spíš naopak.
* Záporný koeficient průměrného věku v modelu explorace je proto **artefakt
  přeplněného modelu**: věk sám s pokrytím nesouvisí (r ≈ 0) a ostatní prediktory
  ho z velké části vysvětlují. Takový koeficient nelze vykládat.
* Dál pracujeme se **standardizovaným pokrytím 45+** (`m_std`), aby výsledky na věku
  nezávisely z konstrukce. Pokrytí 50–69 let (`m_5069`) slouží jako druhá cílová
  proměnná: v kap. 2 se ukáže, že se chová jinak.

## 2 · Kolik rozdílů leží mezi kraji

Explorace uvádí, že „zhruba pětina rozdílů mezi okresy leží mezi kraji“, a čte to
jako stopu organizace péče (zvací systém, kapacity, gynekologové). Podíl
meziskupinového součtu čtverců (η²) ale roste s počtem skupin i bez jakéhokoli
efektu: při 14 krajích a 77 okresech je jeho očekávaná hodnota za nulové
hypotézy (k−1)/(n−1) = 13/76 ≈ **17 %**.

Proto porovnáváme η² s **permutačním testem** (5 000 náhodných přeřazení okresů
do krajů) a počítáme **vnitrotřídní korelaci** (ICC, ANOVA odhad). Ta říká, jaký
podíl rozptylu na kraj skutečně připadá, po odečtení toho, co dělá náhoda.

In [6]:
KRAJE = X.kraj.values
K = pd.Series(KRAJE).nunique()

eta2, icc_anova = an.eta2, an.icc_anova

def kraj_test(y, n_perm=5000):
    return an.kraj_test(y, KRAJE, RNG, n_perm)

CILE_K = {"hrubé pokrytí 45+ (explorace)": X.mamo_2y, **{STD_POPIS[c]: S[c] for c in ["m_std", "m_5069", "p_std"]}}
pasma = P.pivot(index="lau", columns="band", values="m").reindex(LAUS)
CILE_K.update({f"pásmo {b}": pasma[b] for b in BANDS})
KRAJ_T = pd.DataFrame({k: kraj_test(v) for k, v in CILE_K.items()}).T
KRAJ_T.to_csv(OUT / "kraje_rozptyl.csv")
print(f"očekávané η² za nulové hypotézy: (k−1)/(n−1) = {(K-1)/(len(LAUS)-1):.3f}")
KRAJ_T.round(3)

očekávané η² za nulové hypotézy: (k−1)/(n−1) = 0.171


,η²,η² náhodou,p (permutace),ICC
hrubé pokrytí 45+ (explorace),0.224,0.170,0.176,0.069
standardizované 45+,0.250,0.172,0.108,0.102
50–69 let,0.342,0.171,0.006,0.219
"prevence u PL, standardizovaná",0.348,0.171,0.007,0.227
pásmo 45–49,0.267,0.172,0.082,0.124
pásmo 50–54,0.348,0.171,0.006,0.226
pásmo 55–59,0.378,0.170,0.004,0.264
pásmo 60–64,0.340,0.171,0.009,0.216
pásmo 65–69,0.300,0.170,0.028,0.166
pásmo 70–74,0.231,0.170,0.162,0.078


In [7]:
kb = KRAJ_T.loc[[f"pásmo {b}" for b in BANDS]]
kfig = go.Figure()
kfig.add_trace(go.Bar(x=BANDS, y=100 * kb["η²"], name="podíl rozdílů mezi kraji (η²)",
                      marker=dict(color=[S1 if p < 0.05 else GRID for p in kb["p (permutace)"]],
                                  line=dict(color=SURFACE, width=2)),
                      customdata=np.c_[kb["p (permutace)"], kb["ICC"]],
                      hovertemplate=("<b>%{x}</b><br>η² = %{y:.1f} %<br>p = %{customdata[0]:.3f}"
                                     "<br>ICC = %{customdata[1]:.2f}<extra></extra>")))
kfig.add_hline(y=100 * kb["η² náhodou"].mean(), line=dict(color=S2, dash="dash", width=2),
               annotation_text="tolik vyjde čistou náhodou", annotation_position="top left",
               annotation_font=dict(color=S2, size=12))
kfig.update_layout(**layout(
    "Kraj hraje roli hlavně u žen 50–69 let",
    "podíl rozdílů v pokrytí mamografií (2020–2024) mezi kraji, po věkových pásmech · "
    "modře pásma, kde je podíl vyšší než náhodou (permutační p < 0,05)"))
kfig.update_layout(height=460, showlegend=False, yaxis_title="η² [%]",
                   yaxis=dict(range=[0, 60], gridcolor=GRID), margin=dict(l=60, r=30, t=90, b=50))
kfig.write_html(OUT / "kraje_pasma.html", include_plotlyjs="cdn")
kfig

**Jak to číst.**

* U **pokrytí 45+, se kterým pracovala explorace**, je podíl mezi kraji asi 22 %,
  tedy jen o kousek víc, než dá náhoda (17 %). Permutační test ho nepotvrdí
  (p ≈ 0,2) a ICC je nízké (kolem 0,07). Věta „pětina rozdílů leží mezi kraji“
  je v tomhle tvaru **neprůkazná**.
* U žen **50–69 let** je to jinak: mezi kraji leží kolem třetiny rozdílů
  (p < 0,01, ICC ≈ 0,2), v každém pětiletém pásmu 50–69 zvlášť. Stejně tak u
  **prevence u praktika**. Od 70 let krajský vzorec mizí (p > 0,1). Krajská
  (resp. regionální) organizace péče se tedy projevuje u jádra cílové skupiny.
  Ve 45+ celkem ji ředí nejstarší pásma.
* Prakticky: kdo chce mluvit o roli krajů, má mluvit o ženách 50–69 let, ne o 45+
  celkem.

## 3 · Dostupnost a kapacita

### 3.1 Oprava: dojížďka měřená z NRHZS je u řetězců zkreslená

NR-04-86 vykazuje výkon v **okrese zařízení (IČZ)**. U poskytovatelů s více
pracovišti (EUC, AGEL, Medicon…) to může být okres sídla, ne okres, kde mamograf
stojí. Prozradí se to tak, že okres má akreditované pracoviště podle mamo.cz,
ale v datech v něm není vykázaná ani jedna screeningová mamografie. Měřená
dojížďka pak vyjde 100 %, i když ženy nikam nejezdí.

In [8]:
vykonano = toky[toky.rok.isin(nac.ROKY_V)].groupby("cil_lau").mnozstvi.sum() / len(nac.ROKY_V)
fantom = sorted(set(centra.obec_lau) - set(vykonano[vykonano > 0].index))
kam = (toky[toky.rok.isin(nac.ROKY_V) & toky.z_lau.isin(fantom)]
       .sort_values("mnozstvi", ascending=False).groupby("z_lau").head(1)
       .set_index("z_lau").cil_lau.map(OKRES))
print("okresy s pracovištěm, ale bez vykázaných mamografií (výkony se vykazují jinde):")
for l in fantom:
    jm = ", ".join(centra[centra.obec_lau == l].pracoviste.str.slice(0, 45))
    print(f"  {OKRES[l]:<12} dojížďka {X.dojizdka_mamo_pct[l]:5.1f} % → vykázáno hlavně v: "
          f"{kam.get(l, '–'):<16} ({jm})")

X["dojizdka_opr"] = X.dojizdka_mamo_pct.where(~X.index.isin(fantom))
for c, jm in [("dojizdka_mamo_pct", "původní"), ("dojizdka_opr", "bez zkreslených okresů")]:
    g = X[[c]].join(S.m_std).dropna()
    r, p = stats.pearsonr(g[c], g.m_std)
    print(f"dojížďka ({jm:<22}) × standardizované pokrytí: r = {r:+.3f}, p = {p:.3f}, n = {len(g)}")

okresy s pracovištěm, ale bez vykázaných mamografií (výkony se vykazují jinde):
  Benešov      dojížďka 100.0 % → vykázáno hlavně v: Praha            (Medicon services s.r.o. – Mammacentrum Benešo)
  Tachov       dojížďka 100.0 % → vykázáno hlavně v: Plzeň-město      (Poliklinika Tachov – Mamografie)
  Teplice      dojížďka 100.0 % → vykázáno hlavně v: Ústí nad Labem   (EUC Mamocentrum Bílina)
  Česká Lípa   dojížďka 100.0 % → vykázáno hlavně v: Ústí nad Labem   (EUC Mamocentrum Česká Lípa)
  Jeseník      dojížďka 100.0 % → vykázáno hlavně v: Šumperk          (Nemocnice AGEL Jeseník a.s.)
  Vsetín       dojížďka 100.0 % → vykázáno hlavně v: Nový Jičín       (Mammocentrum Vsetín)
  Bruntál      dojížďka 100.0 % → vykázáno hlavně v: Opava            (Mamografické pracoviště Krnov)
dojížďka (původní               ) × standardizované pokrytí: r = -0.340, p = 0.002, n = 77
dojížďka (bez zkreslených okresů) × standardizované pokrytí: r = -0.325, p = 0.006, n = 70


### 3.2 Kapacita v dosahu (2SFCA)

Vzdálenost k *nejbližšímu* pracovišti nebere v úvahu, kolik pracovišť je v dosahu
a kolik žen o ně soupeří. Standardní míra prostorové dostupnosti je
**two-step floating catchment area** (2SFCA):

1. pro každé pracoviště *j*: poměr nabídky k poptávce
   $R_j = S_j / \sum_k P_k\, f(d_{kj})$,
2. pro každý okres *i*: součet poměrů dosažitelných pracovišť
   $A_i = \sum_j R_j\, f(d_{ij})$,

kde $P_k$ jsou ženy 45–74 let v okrese (v jeho těžišti), $f$ je gaussovský
útlum nulový za hranicí dosahu $d_0$ a nabídka $S_j = 1$ za akreditované
pracoviště. Výsledek je **počet pracovišť na 100 000 žen v dosahu**. Dosah
zkoušíme 20, 30 a 45 km.

Nabídku vážit počtem vykázaných výkonů by bylo přesnější, kvůli zkreslení z 3.1
ale nejde. Výkony řetězců by se přičetly okresu sídla.

In [9]:
POPT = (D[D.band.astype(str).isin(nac.BANDS[:6]) & D.rok.isin(nac.ROKY_V)]
        .groupby("lau").zeny.mean().reindex(LAUS))          # ženy 45–74, roční průměr
clon = np.array([CENTROID[l][0] for l in LAUS]); clat = np.array([CENTROID[l][1] for l in LAUS])
DIST = nac.haversine(clon[:, None], clat[:, None], centra.lon.values[None, :], centra.lat.values[None, :])

def sfca(d0, nabidka=None):
    '''2SFCA (utils/analyza.sfca, včetně kontroly součtu nabídky): pracoviště na 100 000 žen.'''
    return pd.Series(an.sfca(DIST, POPT.values, d0, nabidka), index=LAUS)

for d0 in (20, 30, 45):
    X[f"sfca_{d0}"] = sfca(d0)
    POPIS[f"sfca_{d0}"] = (f"pracoviště v dosahu {d0} km (2SFCA)", "na 100 000 žen")

rows = []
for c in ["sfca_20", "sfca_30", "sfca_45", "vzdalenost_km", "ma_mamocentrum", "dojizdka_opr"]:
    g = X[[c]].join(S[["m_std", "m_5069"]]).dropna()
    for cil in ("m_std", "m_5069"):
        r, p = stats.pearsonr(g[c], g[cil])
        rows.append({"ukazatel": POPIS.get(c, ("dojížďka mimo okres, opravená", ""))[0],
                     "cíl": STD_POPIS[cil], "n": len(g), "r": r, "p": p,
                     "Spearman ρ": stats.spearmanr(g[c], g[cil])[0]})
DOST = pd.DataFrame(rows)
DOST.to_csv(OUT / "dostupnost.csv", index=False)
print(f"okresů bez pracoviště v dosahu: 20 km {int((X.sfca_20 == 0).sum())}, "
      f"30 km {int((X.sfca_30 == 0).sum())}, 45 km {int((X.sfca_45 == 0).sum())}")
DOST.round(3)

okresů bez pracoviště v dosahu: 20 km 15, 30 km 5, 45 km 0


,ukazatel,cíl,n,r,p,Spearman ρ
0,pracoviště v dosahu 20 km (2SFCA),standardizované 45+,77,0.055,0.637,0.152
1,pracoviště v dosahu 20 km (2SFCA),50–69 let,77,-0.004,0.972,0.089
2,pracoviště v dosahu 30 km (2SFCA),standardizované 45+,77,0.025,0.830,0.044
3,pracoviště v dosahu 30 km (2SFCA),50–69 let,77,-0.032,0.784,0.008
4,pracoviště v dosahu 45 km (2SFCA),standardizované 45+,77,0.041,0.723,0.010
5,pracoviště v dosahu 45 km (2SFCA),50–69 let,77,-0.006,0.959,0.002
6,vzdálenost k nejbližšímu pracovišti,standardizované 45+,77,-0.129,0.265,-0.180
7,vzdálenost k nejbližšímu pracovišti,50–69 let,77,-0.026,0.821,-0.081
8,okres má mamografické pracoviště,standardizované 45+,77,0.233,0.041,0.228
9,okres má mamografické pracoviště,50–69 let,77,0.156,0.175,0.143


In [10]:
smap = go.Figure()
smap.add_trace(go.Choropleth(
    geojson=GEO, locations=LAUS, z=X.sfca_30, colorscale=BLUES,
    marker_line_color=SURFACE, marker_line_width=0.8,
    customdata=np.c_[X.okres, [cz(v, 2) for v in X.sfca_30], [cz(v) for v in S.m_std]],
    hovertemplate=("<b>%{customdata[0]}</b><br>pracovišť v dosahu 30 km: %{customdata[1]} "
                   "na 100 000 žen<br>standardizované pokrytí: %{customdata[2]} %<extra></extra>"),
    colorbar=skala("pracoviště<br>na 100 000 žen", suffix="")))
smap.add_trace(go.Scattergeo(lon=centra.lon, lat=centra.lat, mode="markers",
                             marker=dict(size=7, color=S2, line=dict(color=SURFACE, width=1.5)),
                             text=centra.pracoviste, hoverinfo="text", name="pracoviště"))
smap.update_layout(**layout("Kapacita mamografických pracovišť v dosahu 30 km",
                            "2SFCA · nabídka = akreditovaná pracoviště, poptávka = ženy 45–74 let"))
mapa_geo(smap)
smap.update_layout(height=560, showlegend=False, margin=dict(l=10, r=10, t=90, b=20))
smap.write_html(OUT / "mapa_2sfca.html", include_plotlyjs="cdn")
smap

**Jak to číst.**

* **Geografická dostupnost pokrytí nevysvětluje, ani s lepší mírou.** Kapacita v
  dosahu (2SFCA) s pokrytím nesouvisí při žádném dosahu (|r| < 0,1), u 45+ ani u
  50–69 let. Vzdálenost k nejbližšímu pracovišti také ne.
* **Měřená dojížďka ale souvisí, a nejde o artefakt.** Sedm okresů s vlastním
  pracovištěm mělo dojížďku 100 % jen proto, že řetězec vykazuje výkony jinde.
  Po jejich vyřazení vztah drží (r ≈ −0,33, p < 0,01). Slabší je u 50–69 let.
  Podobně okresy s vlastním pracovištěm mají pokrytí o něco vyšší (r ≈ 0,23).
* Rozdíl mezi oběma zjištěními je poučný. Rozhoduje, **jestli ženy jezdí jinam**,
  ne kolik pracovišť je v dosahu na mapě. Dojížďka je ovšem sama výsledkem
  chování: kde místní pracoviště nestačí nebo nemá dobrou pověst, ženy buď jezdí
  jinam, nebo nejdou vůbec. Kauzálně ji proto číst nelze. Co měřit neumíme:
  objednací lhůty, ordinační hodiny, dopravní spojení.

## 4 · Model postavený na kauzálním schématu

Model explorace míchal tři druhy proměnných, které do jedné regrese nepatří:

```
  PŘÍČINY (determinanty)                   SOUBĚŽNÉ CHOVÁNÍ
  socioekonomika, urbanizace  ──┐          gynekologická prevence,
  dostupnost, kapacita        ──┼──► POKRYTÍ ◄──  cytologie, test OK
  kraj (organizace péče)      ──┘     │      (společná příčina: „chodí na prevenci“)
                                      ▼
                             DŮSLEDKY (do modelu NE)
                             léčená rakovina prsu, nezhoubné nálezy,
                             diagnostické zobrazení prsu
```

* **Důsledky** screeningu (screening nádory a nálezy *najde*) jako prediktory
  vyřazujeme. Vztah s pokrytím mají obráceně.
* **Souběžné chování** shrnujeme do jednoho **indexu preventivního chování**
  (průměr z-skóre gynekologické prohlídky, cytologie a testu na okultní krvácení).
  Tři silně korelované ukazatele téhož by si v modelu jen přetahovaly efekt.
* Modely skládáme postupně: **M1** socioekonomika a urbanizace → **M2** +
  dostupnost → **M3** + preventivní chování. Rozhodující měřítko je
  **vysvětlený rozptyl mimo vzorek** (křížová validace), ne R² na trénovacích
  datech.

In [11]:
PREV3 = an.PREV3
X["prev_index"] = an.prev_index(X)
POPIS["prev_index"] = ("index preventivního chování", "z-skóre")
cv_ = X[PREV3].cov().values
alfa = 3 / 2 * (1 - np.trace(cv_) / cv_.sum())
print(f"Cronbachovo α indexu: {alfa:.2f}  (korelace složek: "
      + ", ".join(f"{POPIS[a][0][:20]}×{POPIS[b][0][:12]} {X[a].corr(X[b]):.2f}"
                  for a, b in [(PREV3[0], PREV3[1]), (PREV3[0], PREV3[2]), (PREV3[1], PREV3[2])]) + ")")

M1 = an.M1
M2 = M1 + ["sfca_30", "ma_mamocentrum"]
M3 = M2 + ["prev_index"]
MODELY = {"M1 socioekonomika": M1, "M2 + dostupnost": M2, "M3 + prevence": M3,
          "jen index prevence": ["prev_index"], "model explorace (22)": OLD}

def cv_r2(cols, y, n_rep=50):
    '''R² mimo vzorek: opakovaná 5násobná křížová validace (50×) a leave-one-out.'''
    return an.cv_r2(X[cols].values, y.values, n_rep)

rows = []
for cil in ("m_std", "m_5069"):
    y = S[cil]
    for nm, cols in MODELY.items():
        o_ = sm.OLS(y.values, sm.add_constant(X[cols].values)).fit()
        r5, rl = cv_r2(cols, y)
        rows.append({"cíl": STD_POPIS[cil], "model": nm, "prediktorů": len(cols),
                     "R²": o_.rsquared, "adj. R²": o_.rsquared_adj,
                     "R² 5-fold CV": r5, "R² LOO": rl})
FITY = pd.DataFrame(rows)
FITY.to_csv(OUT / "modely_fit.csv", index=False)
FITY.round(3)

Cronbachovo α indexu: 0.58  (korelace složek: preventivní gynekolo×cytologie dě 0.59, preventivní gynekolo×test na okul 0.50, cytologie děložního ×test na okul 0.37)


,cíl,model,prediktorů,R²,adj. R²,R² 5-fold CV,R² LOO
0,standardizované 45+,M1 socioekonomika,6,0.220,0.153,0.067,0.080
1,standardizované 45+,M2 + dostupnost,8,0.311,0.230,0.062,0.099
2,standardizované 45+,M3 + prevence,9,0.499,0.431,0.288,0.321
3,standardizované 45+,jen index prevence,1,0.270,0.260,0.213,0.219
4,standardizované 45+,model explorace (22),22,0.677,0.545,0.244,0.325
5,50–69 let,M1 socioekonomika,6,0.315,0.256,0.186,0.198
6,50–69 let,M2 + dostupnost,8,0.387,0.314,0.171,0.203
7,50–69 let,M3 + prevence,9,0.543,0.482,0.361,0.390
8,50–69 let,jen index prevence,1,0.261,0.251,0.206,0.212
9,50–69 let,model explorace (22),22,0.699,0.576,0.299,0.369


In [12]:
f0 = FITY[FITY["cíl"] == STD_POPIS["m_std"]]
mfig = go.Figure()
for col, barva, jm in [("R²", GRID, "R² na trénovacích datech"),
                       ("R² 5-fold CV", S1, "R² mimo vzorek (křížová validace)")]:
    mfig.add_trace(go.Bar(x=f0.model, y=f0[col], name=jm,
                          marker=dict(color=barva, line=dict(color=SURFACE, width=2)),
                          text=[cz(v, 2) for v in f0[col]], textposition="outside",
                          hovertemplate="<b>%{x}</b><br>%{y:.3f}<extra>" + jm + "</extra>"))
mfig.update_layout(**layout(
    "Mimo vzorek vysvětluje pokrytí hlavně preventivní chování",
    "standardizované pokrytí 45+, 77 okresů · šedě R² na datech, na kterých se model učil, "
    "modře R² na okresech, které model neviděl"))
mfig.update_layout(height=460, barmode="group", bargap=0.3, yaxis=dict(range=[0, 0.8], gridcolor=GRID),
                   legend=dict(orientation="h", y=-0.15, x=0), margin=dict(l=60, r=30, t=90, b=90))
mfig.write_html(OUT / "modely_r2.html", include_plotlyjs="cdn")
mfig

### 4.1 Koeficienty M3

Standardizované koeficienty (změna pokrytí v p. b. na jednu směrodatnou odchylku
prediktoru) se dvěma odhady: **OLS s robustními chybami HC3** a **smíšený model
s náhodným interceptem kraje**. Ten zohledňuje, že okresy v témže kraji nejsou
nezávislé.

In [13]:
def standardizuj_x(cols, df=X):
    return (df[cols] - df[cols].mean()) / df[cols].std()

def fit_m3(y, df=X, idx=None):
    idx = df.index if idx is None else idx
    Z = standardizuj_x(M3, df.loc[idx])
    ols = sm.OLS(y.loc[idx].values, sm.add_constant(Z.values)).fit(cov_type="HC3")
    return ols, Z

ols3, Z3 = fit_m3(S.m_std)
mix_df = Z3.assign(y=S.m_std.values, kraj=X.kraj.values)
mix3 = smf.mixedlm("y ~ " + " + ".join(M3), mix_df, groups="kraj").fit(reml=True, method="powell")   # lbfgs zde končí singulární maticí
KOEF3 = pd.DataFrame({
    "β OLS": ols3.params[1:], "CI OLS": [tuple(r) for r in ols3.conf_int()[1:]], "p OLS": ols3.pvalues[1:],
    "β smíšený": mix3.params[M3].values, "p smíšený": mix3.pvalues[M3].values,
    "VIF": vif(M3).values}, index=[POPIS[c][0] for c in M3])
var_kraj = float(mix3.cov_re.iloc[0, 0])
print(f"smíšený model: rozptyl kraje {var_kraj:.2f}, reziduální {mix3.scale:.2f} "
      f"→ ICC po zohlednění prediktorů {var_kraj / (var_kraj + mix3.scale):.2f}")
KOEF3.drop(columns="CI OLS").round(3)

smíšený model: rozptyl kraje 2.00, reziduální 10.36 → ICC po zohlednění prediktorů 0.16


,β OLS,p OLS,β smíšený,p smíšený,VIF
"vysokoškolačky, ženy 15+",0.923,0.171,0.713,0.310,2.808
podíl nezaměstnaných žen,-0.494,0.592,-1.057,0.208,3.791
příspěvek na bydlení na obyvatele,-0.726,0.437,-0.311,0.739,4.918
ženy s cizím občanstvím,-0.642,0.254,-0.890,0.109,1.662
obyvatelé obcí do 499 obyv.,1.272,0.069,1.084,0.067,2.028
obyvatelé obcí 50 000+ obyv.,1.005,0.171,1.127,0.099,3.049
pracoviště v dosahu 30 km (2SFCA),-0.681,0.357,-0.558,0.297,1.787
okres má mamografické pracoviště,1.282,0.057,1.341,0.013,1.967
index preventivního chování,2.347,0.000,2.510,0.000,1.404


In [14]:
cfig = go.Figure()
ci = np.array(KOEF3["CI OLS"].tolist())
cfig.add_trace(go.Scatter(x=KOEF3["β OLS"], y=KOEF3.index, mode="markers", name="OLS (HC3)",
                          marker=dict(size=10, color=S1),
                          error_x=dict(type="data", symmetric=False, array=ci[:, 1] - KOEF3["β OLS"],
                                       arrayminus=KOEF3["β OLS"] - ci[:, 0], color=S1, thickness=1.5),
                          hovertemplate="<b>%{y}</b><br>β = %{x:.2f} p. b.<extra>OLS</extra>"))
cfig.add_trace(go.Scatter(x=KOEF3["β smíšený"], y=KOEF3.index, mode="markers",
                          name="smíšený model (kraj)", marker=dict(size=10, color=S2, symbol="diamond"),
                          hovertemplate="<b>%{y}</b><br>β = %{x:.2f} p. b.<extra>smíšený</extra>"))
cfig.add_vline(x=0, line=dict(color=AXIS, width=1))
cfig.update_layout(**layout("Model M3: o kolik p. b. se pokrytí liší na 1 SD prediktoru",
                            "standardizované pokrytí 45+ · 95% interval spolehlivosti (HC3)"))
cfig.update_layout(height=470, margin=dict(l=260, r=30, t=90, b=80), xaxis_title="β [p. b. / SD]",
                   yaxis=dict(autorange="reversed", showgrid=False, tickfont=dict(color=INK2)),
                   legend=dict(orientation="h", y=-0.15, x=0))
cfig.write_html(OUT / "m3_koeficienty.html", include_plotlyjs="cdn")
cfig

### 4.2 Kontrola výběrem proměnných (lasso) a vlivnými okresy

Pořadí M1 → M3 jsme zvolili my. Jako nezávislou kontrolu necháme lasso vybrat
z **všech** kandidátů kromě důsledků screeningu (a zkreslené dojížďky z 3.1).
Abychom nevěřili jednomu náhodnému výběru, opakujeme to na 200 bootstrapových
vzorcích okresů a díváme se, **jak často** se který prediktor dostane do modelu
(stability selection).

In [15]:
DUSLEDKY = an.DUSLEDKY
KAND = [c for c in POPIS if c in X.columns and c not in DUSLEDKY
        and c not in ("dojizdka_mamo_pct", "prev_index", "sfca_20", "sfca_45")]
Zk, yk = X[KAND].values, S.m_std.values
vyber = np.zeros(len(KAND))
N_BOOT = 200
for b in range(N_BOOT):
    i = RNG.integers(0, len(yk), len(yk))
    las = make_pipeline(StandardScaler(), LassoCV(cv=5, alphas=40, max_iter=20000, random_state=b))
    las.fit(Zk[i], yk[i])
    vyber += las[-1].coef_ != 0
STAB = pd.Series(vyber / N_BOOT, index=[POPIS[c][0] for c in KAND]).sort_values()
STAB.to_csv(OUT / "lasso_stabilita.csv")

lfig = go.Figure(go.Bar(x=STAB.values, y=STAB.index, orientation="h",
                        marker=dict(color=[S1 if v >= 0.6 else GRID for v in STAB.values]),
                        hovertemplate="<b>%{y}</b><br>vybrán v %{x:.0%} vzorků<extra></extra>"))
lfig.add_vline(x=0.6, line=dict(color=S2, dash="dash"))
lfig.update_layout(**layout("Které prediktory lasso vybírá opakovaně",
                            f"podíl z {N_BOOT} bootstrapových vzorků, kdy lasso prediktor ponechalo · "
                            f"cíl: standardizované pokrytí 45+ · modře ≥ 60 %"))
lfig.update_layout(height=max(560, 22 * len(KAND) + 160), margin=dict(l=280, r=30, t=90, b=40),
                   xaxis=dict(range=[0, 1], tickformat=".0%", gridcolor=GRID),
                   yaxis=dict(tickfont=dict(size=11, color=INK2), showgrid=False))
lfig.write_html(OUT / "lasso_stabilita.html", include_plotlyjs="cdn")
lfig

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.434335e-01, tolerance: 1.219e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.275538e-01, tolerance: 8.812e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.295871e-01, tolerance: 1.154e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.165993e-01, tolerance: 1.154e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.384841e-01, tolerance: 1.401e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.332196e-01, tolerance: 9.221e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.033268e-01, tolerance: 1.207e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.730990e-01, tolerance: 1.041e-01
  model = cd_fast.enet_coordinate_descent_gram(


C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.459277e-01, tolerance: 9.993e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.132404e-01, tolerance: 9.993e-02
  model = cd_fast.enet_coordinate_descent_gram(


...

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.377490e-01, tolerance: 9.701e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.010765e-01, tolerance: 9.701e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.444952e-01, tolerance: 1.503e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.638437e-01, tolerance: 1.503e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.360072e-01, tolerance: 8.787e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.762939e-01, tolerance: 8.787e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.285952e-01, tolerance: 7.315e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.172437e+00, tolerance: 9.364e-02
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the

C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.972434e-01, tolerance: 1.494e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\vittu\src\mamograf-data-exploration\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.984446e+00, tolerance: 1.494e-01
  model = cd_fast.enet_coordinate_descent_gram(


In [16]:
ols3_plain = sm.OLS(S.m_std.values, sm.add_constant(Z3.values)).fit()
cook = pd.Series(ols3_plain.get_influence().cooks_distance[0], index=LAUS)
print("nejvlivnější okresy v M3 (Cookova vzdálenost):")
print(cook.sort_values(ascending=False).head(5).rename(OKRES).round(3).to_string())

MESTA = ["CZ0100", "CZ0642", "CZ0806", "CZ0323"]          # Praha, Brno, Ostrava, Plzeň
bez = [l for l in LAUS if l not in MESTA]
ols_bez, _ = fit_m3(S.m_std, idx=bez)
i_prev = M3.index("prev_index") + 1
print(f"\nindex prevence v M3: β = {ols3.params[i_prev]:.2f} (p = {ols3.pvalues[i_prev]:.4f}); "
      f"bez čtyř velkých měst β = {ols_bez.params[i_prev]:.2f} (p = {ols_bez.pvalues[i_prev]:.4f})")

nejvlivnější okresy v M3 (Cookova vzdálenost):
Jeseník          0.144
Frýdek-Místek    0.118
Bruntál          0.096
Kroměříž         0.086
Most             0.084

index prevence v M3: β = 2.35 (p = 0.0001); bez čtyř velkých měst β = 2.30 (p = 0.0000)


**Jak to číst.**

* **Starý model je silně přeučený.** Na trénovacích datech vysvětlí asi dvě
  třetiny rozptylu, na okresech, které neviděl, jen kolem čtvrtiny. Rozdíl mezi
  oběma čísly je přesně to, co model „vysvětlil“ šumem.
* **Socioekonomika u 45+ sama nevysvětluje skoro nic** (M1: R² mimo vzorek kolem
  0,07). U 50–69 let o něco víc (kolem 0,19). Dostupnost (M2) nepřidá nic. Teprve
  **preventivní chování** posune vysvětlený rozptyl na zhruba 0,3 (45+), resp.
  0,36 (50–69). Mimo vzorek je to stejně jako model s 22 prediktory, jenže s 9
  prediktory a bez proměnných s obrácenou příčinností.
* Index preventivního chování je nejsilnější koeficient M3 (≈ +2,3 p. b. na SD),
  průkazný v OLS i ve smíšeném modelu a stabilní i bez velkých měst. Lasso
  ponechává všechny tři jeho složky v 84–95 % bootstrapových vzorků.
* Druhý stabilní prediktor je **vlastní pracoviště v okrese** (≈ +1,3 p. b.,
  ve smíšeném modelu p ≈ 0,01, v lassu 91 % vzorků). Ladí to s kap. 3:
  nerozhoduje kapacita v dosahu, ale jestli je mamograf přímo v okrese.
* Lasso s penalizací volenou křížovou validací vybírá štědře, takže se často
  objeví i další ukazatele (volby, velikost obcí, CHOPN). Žádný z nich ale nemá
  v M1–M3 stabilní efekt. Berte je jako tipy, ne jako zjištění.
* Smíšený model nechává mezi kraji i po zohlednění prediktorů nezanedbatelný
  rozptyl. Krajský vzorec z kap. 2 tedy prediktory nevysvětlí.
* Pozor na výklad: index prevence není *příčina* pokrytí, ale souběžné chování
  se společnou příčinou (ochota chodit na prevenci, aktivita gynekologů). Model
  říká, že **rozdíly v mamografii jsou součástí širšího vzorce preventivního
  chování žen**, ne že by gynekologická prohlídka „způsobovala“ mamografii.

## 5 · Prostorová závislost

Sousední okresy si bývají podobné (sdílejí pracoviště, pojišťovny, gynekology).
Pak nejsou 77 nezávislých pozorování a *p* hodnoty vycházejí menší, než by měly.
Měříme to **Moranovým I** se sousedstvím podle společné hranice (matice
sousedství z geometrie RÚIAN, řádkově normalizovaná). Hodnota kolem
−1/(n−1) ≈ −0,013 znamená žádnou závislost, kladná znamená shluky podobných
okresů. *p* je z permutačního testu (999 přeřazení).

In [17]:
def sousedstvi(tol=0.004, min_bodu=2):
    return an.sousedstvi(GEO, LAUS, tol, min_bodu)

WB = sousedstvi()
WR = WB / WB.sum(1, keepdims=True)
print(f"sousedů na okres: průměr {WB.sum(1).mean():.1f}, rozpětí {WB.sum(1).min():.0f}–{WB.sum(1).max():.0f}")

def moran(y, W=WR, n_perm=999):
    return an.moran(y, W, RNG, n_perm)

rows = []
for nm, y in [("standardizované pokrytí 45+", S.m_std), ("pokrytí 50–69 let", S.m_5069),
              ("prevence u PL (standardizovaná)", S.p_std), ("index prevence", X.prev_index)]:
    rows.append(dict(zip(["proměnná", "Moran I", "p", "I náhodou"], [nm, *moran(y)])))
for nm, cols in list(MODELY.items())[:3]:
    res = sm.OLS(S.m_std.values, sm.add_constant(X[cols].values)).fit().resid
    rows.append(dict(zip(["proměnná", "Moran I", "p", "I náhodou"], [f"rezidua {nm}", *moran(res)])))
MORAN = pd.DataFrame(rows).set_index("proměnná")
MORAN.to_csv(OUT / "moran.csv")
MORAN.round(3)

sousedů na okres: průměr 4.9, rozpětí 1–8


,Moran I,p,I náhodou
proměnná,,,
standardizované pokrytí 45+,0.198,0.004,-0.014
pokrytí 50–69 let,0.301,0.001,-0.016
prevence u PL (standardizovaná),0.294,0.001,-0.015
index prevence,0.167,0.005,-0.018
rezidua M1 socioekonomika,0.051,0.187,-0.013
rezidua M2 + dostupnost,0.062,0.153,-0.015
rezidua M3 + prevence,0.139,0.023,-0.016


Rezidua M3 zůstávají prostorově závislá. Jako citlivostní test proto přidáme do
M3 **Moranovy vlastní vektory** (spatial eigenvector filtering): vlastní vektory
dvojitě centrované matice sousedství popisují prostorové vzory od celostátních po
regionální. Přidáváme je postupně, dokud rezidua nepřestanou být závislá
(p > 0,1, nejvýš 10 vektorů). Zajímá nás, jestli se změní koeficient indexu
prevence a jeho chyba.

In [18]:
n_ = len(LAUS)
Mc = np.eye(n_) - np.ones((n_, n_)) / n_
WS = (WB + WB.T) / 2
ev, EV = np.linalg.eigh(Mc @ WS @ Mc)
poradi = np.argsort(ev)[::-1]
EV = EV[:, poradi][:, ev[poradi] > 0]

Zf, pridane = Z3.copy(), 0
for k in range(10):
    res = sm.OLS(S.m_std.values, sm.add_constant(Zf.values)).fit().resid
    if moran(res, n_perm=499)[1] > 0.1:
        break
    # vybrat vektor, který nejvíc sníží Moran I reziduí
    kand = []
    for j in range(min(30, EV.shape[1])):
        if f"ev{j}" in Zf:
            continue
        r_ = sm.OLS(S.m_std.values, sm.add_constant(np.c_[Zf.values, EV[:, j]])).fit().resid
        z_ = r_ - r_.mean()
        kand.append((n_ / WR.sum() * (z_ @ WR @ z_) / (z_ @ z_), j))
    j = min(kand)[1]
    Zf[f"ev{j}"] = EV[:, j]
    pridane += 1
ols_f = sm.OLS(S.m_std.values, sm.add_constant(Zf.values)).fit(cov_type="HC3")
res_f = ols_f.resid
print(f"přidáno vlastních vektorů: {pridane}; Moran I reziduí {moran(res_f)[0]:.3f} (p = {moran(res_f)[1]:.3f})")
print(f"index prevence: β = {ols3.params[i_prev]:.2f} ± {ols3.bse[i_prev]:.2f} (bez filtru) → "
      f"{ols_f.params[i_prev]:.2f} ± {ols_f.bse[i_prev]:.2f} (s filtrem), p = {ols_f.pvalues[i_prev]:.4f}")

přidáno vlastních vektorů: 1; Moran I reziduí 0.021 (p = 0.296)


index prevence: β = 2.35 ± 0.59 (bez filtru) → 2.35 ± 0.56 (s filtrem), p = 0.0000


In [19]:
z = (S.m_std - S.m_std.mean()) / S.m_std.std()
lag = WR @ z.values
mor = go.Figure()
mor.add_trace(go.Scatter(x=z, y=lag, mode="markers", marker=dict(size=9, color=S1, line=dict(color=SURFACE, width=1.5)),
                         customdata=np.c_[S.okres, S.kraj],
                         hovertemplate="<b>%{customdata[0]}</b> · %{customdata[1]}<br>okres %{x:.2f} SD"
                                       "<br>průměr sousedů %{y:.2f} SD<extra></extra>"))
k_ = np.polyfit(z, lag, 1); xs = np.array([z.min(), z.max()])
mor.add_trace(go.Scatter(x=xs, y=np.polyval(k_, xs), mode="lines", line=dict(color=MUTED, width=2), hoverinfo="skip"))
I0, p0, _ = MORAN.loc["standardizované pokrytí 45+"]
mor.update_layout(**layout("Okresy s vysokým pokrytím sousedí s okresy s vysokým pokrytím",
                           f"Moranův diagram · standardizované pokrytí 45+ · I = {cz(I0, 2)}, p = {cz(p0, 3)}"))
mor.update_layout(height=500, showlegend=False, xaxis_title="pokrytí okresu [SD]",
                  yaxis_title="průměrné pokrytí sousedů [SD]", margin=dict(l=60, r=30, t=90, b=60))
mor.write_html(OUT / "moran.html", include_plotlyjs="cdn")
mor

**Jak to číst.**

* Pokrytí je **prostorově shlukované**: kladné Moranovo I, výraznější u 50–69 let
  a u prevence u praktika. Odpovídá to kap. 2: regionální vzorec je skutečný,
  jen nejde přesně po hranicích krajů.
* Rezidua M1 a M2 už shlukovaná nejsou. Socioekonomika a urbanizace jsou
  prostorově uspořádané a prostorovou část pokrytí pohltí, i když celkově
  vysvětlí málo. Po přidání indexu prevence se shlukování v reziduích vrátí:
  index vysvětlí hlavně „neprostorovou“ část rozdílů a regionální složka zbude.
* Jeden Moranův vlastní vektor zbytkové shlukování odstraní. Koeficient indexu
  prevence se přitom nezmění a zůstane průkazný. Hlavní závěr kap. 4 tedy
  prostorovou závislostí nevzniká.
* Pro ostatní korelace v exploraci (kap. 7–10) platí, že jejich *p* hodnoty jsou
  o něco optimističtější, než vypadají. Efektivní počet nezávislých okresů je
  menší než 77.

## 6 · Panel: změny uvnitř okresů

Průřezové korelace srovnávají okresy mezi sebou a mohou je zkreslit trvalé
rozdíly (struktura poskytovatelů, historie). Panel se ptá jinak: **když se v
okrese něco změní, změní se i pokrytí?** Pevné efekty okresu odečtou všechno
trvalé, pevné efekty roku celostátní výkyvy (covid 2020).

Pracujeme s **ročním** věkově standardizovaným pokrytím (dvouletá okna by se
mezi roky překrývala).

In [20]:
ROCNI = an.rocni_std(D, STANDARD)    # roční standardizované pokrytí (nulové buňky ÚZIS → 0)
SIG = ROCNI.groupby("rok").m1.agg(["mean", "std"])
SIG["cv"] = SIG["std"] / SIG["mean"]
print("σ-konvergence (roční standardizované pokrytí 45+):")
print(SIG.T.round(2).to_string())
w_ = ROCNI.pivot(index="lau", columns="rok", values="m1")
print("\nstálost pořadí okresů (Spearman): "
      + ", ".join(f"{a}×{b} {stats.spearmanr(w_[a], w_[b])[0]:.2f}"
                  for a, b in [(2011, 2024), (2014, 2024), (2019, 2024)]))

σ-konvergence (roční standardizované pokrytí 45+):
rok    2010   2011   2012   2013   2014   2015   2016   2017   2018   2019  2020   2021   2022   2023   2024
mean  20.78  21.41  23.13  24.60  26.12  25.69  26.23  26.36  26.25  26.84  24.6  27.29  26.91  27.77  28.13
std    4.30   3.76   3.09   3.18   2.74   2.94   2.92   2.55   2.61   2.52   2.5   2.75   2.31   2.74   2.33
cv     0.21   0.18   0.13   0.13   0.10   0.11   0.11   0.10   0.10   0.09   0.1   0.10   0.09   0.10   0.08

stálost pořadí okresů (Spearman): 2011×2024 0.45, 2014×2024 0.57, 2019×2024 0.77


In [21]:
wide = ROCNI.pivot(index="lau", columns="rok", values="m1")
pfig = go.Figure()
for l in LAUS:
    pfig.add_trace(go.Scatter(x=wide.columns, y=wide.loc[l], mode="lines",
                              line=dict(color=GRID, width=1), hoverinfo="text",
                              text=OKRES[l], showlegend=False))
pfig.add_trace(go.Scatter(x=SIG.index, y=SIG["mean"], mode="lines+markers", name="průměr okresů",
                          line=dict(color=S1, width=3)))
pfig.add_trace(go.Scatter(x=SIG.index, y=100 * SIG["cv"], mode="lines+markers", yaxis="y2",
                          name="variační koeficient mezi okresy", line=dict(color=S2, width=2, dash="dash")))
pfig.update_layout(**layout("Okresy se k sobě přibližují",
                            "roční standardizované pokrytí 45+ (šedě jednotlivé okresy) · "
                            f"variační koeficient (pravá osa) klesl z {cz(100*SIG.cv.iloc[0], 0)} % "
                            f"na {cz(100*SIG.cv.iloc[-1], 0)} %"))
pfig.update_layout(height=500, yaxis_title="pokrytí za rok [%]",
                   yaxis2=dict(overlaying="y", side="right", title="variační koeficient [%]",
                               range=[0, 25], showgrid=False, tickfont=dict(color=MUTED)),
                   legend=dict(orientation="h", y=-0.12, x=0), margin=dict(l=60, r=60, t=90, b=80))
pfig.write_html(OUT / "panel_konvergence.html", include_plotlyjs="cdn")
pfig

### 6.1 Covidový propad

Propad 2020 proti 2019 a dohnání v letech 2021–2022 proti 2018–2019 (tedy dva
roky, aby se dohnalo i odložené vyšetření z dvouletého cyklu).

In [22]:
covid = pd.DataFrame({"propad_2020_pct": 100 * (wide[2020] / wide[2019] - 1),
                      "dohnani_2122_pct": 100 * ((wide[2021] + wide[2022]) / (wide[2018] + wide[2019]) - 1)})
print(f"propad 2020: průměr {covid.propad_2020_pct.mean():.1f} %, SD {covid.propad_2020_pct.std():.1f} "
      f"(od {covid.propad_2020_pct.min():.0f} do {covid.propad_2020_pct.max():.0f} %)")
print(f"dohnání 2021–22: průměr {covid.dohnani_2122_pct.mean():+.1f} %, SD {covid.dohnani_2122_pct.std():.1f}")
rows = []
for c in ["sfca_30", "ma_mamocentrum", "obce_4_pct", "vs_zeny_pct", "nezam_zeny_pct", "prev_index"]:
    for k in covid:
        r, p = stats.pearsonr(X[c], covid[k])
        rows.append({"ukazatel": POPIS[c][0], "co": k, "r": r, "p": p})
COV = pd.DataFrame(rows).pivot(index="ukazatel", columns="co", values=["r", "p"])
COV.round(3)

propad 2020: průměr -8.1 %, SD 7.5 (od -42 do 15 %)
dohnání 2021–22: průměr +2.2 %, SD 4.2


r                  \
co                                dohnani_2122_pct propad_2020_pct   
ukazatel                                                             
index preventivního chování                 -0.051           0.053   
obyvatelé obcí 50 000+ obyv.                -0.012           0.097   
okres má mamografické pracoviště            -0.092          -0.131   
podíl nezaměstnaných žen                    -0.195          -0.053   
pracoviště v dosahu 30 km (2SFCA)           -0.221          -0.233   
vysokoškolačky, ženy 15+                     0.119           0.128   

                                                 p                  
co                                dohnani_2122_pct propad_2020_pct  
ukazatel                                                            
index preventivního chování                  0.663           0.650  
obyvatelé obcí 50 000+ obyv.                 0.918           0.402  
okres má mamografické pracoviště             0.424           0.255  
podíl nezaměstnaných žen                     0.090           0.646  
pracoviště v dosahu 30 km (2SFCA)            0.053           0.041  
vysokoškolačky, ženy 15+                     0.301           0.268

### 6.2 Pevné efekty: jdou spolu změny prevence a mamografie?

Logaritmus počtu mamografií v okrese a roce vysvětlujeme logaritmem počtu
pacientek s gynekologickou prevencí (a případně cytologií a testem OK), s
kontrolou počtu žen 45+ a s pevnými efekty okresu i roku. Koeficient je
**elasticita**: o kolik % se změní počet mamografií, když v témže okrese
vzroste gynekologická prevence o 1 %. Chyby jsou shlukované po okresech. NRHZS
pokrývá 2019–2024. Jako kontrolu vynecháváme covidové roky 2020–2021.

Pro srovnání stejný model s **prevencí u praktika** (PPS-08-04, 2011–2024).

In [23]:
nrw = (nr[nr.ukazatel.isin(["gyn_prevence", "cytologie", "toks"])]
       .pivot_table(index=["lau", "rok"], columns="ukazatel", values="hodnota").reset_index())
T = D[D.band == TOTAL][["lau", "rok", "pocet_vysetreni", "zeny"]].merge(nrw, on=["lau", "rok"])
for c in ["pocet_vysetreni", "zeny", "gyn_prevence", "cytologie", "toks"]:
    T[f"l_{c}"] = np.log(T[c])

def fe(formule, df):
    m = smf.ols(formule + " + C(lau) + C(rok)", df).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(df.lau)[0]})
    return m

rows = []
for nm, f_, df in [
        ("gyn. prevence, 2019–2024", "l_pocet_vysetreni ~ l_gyn_prevence + l_zeny", T),
        ("gyn. prevence, bez 2020–21", "l_pocet_vysetreni ~ l_gyn_prevence + l_zeny", T[~T.rok.isin([2020, 2021])]),
        ("gyn. + cytologie + test OK", "l_pocet_vysetreni ~ l_gyn_prevence + l_cytologie + l_toks + l_zeny", T)]:
    m = fe(f_, df)
    for v in [c for c in m.params.index if c.startswith("l_") and c != "l_zeny"]:
        rows.append({"model": nm, "proměnná": v[2:], "elasticita": m.params[v],
                     "SE (shluky)": m.bse[v], "p": m.pvalues[v], "n": int(m.nobs)})
Q = ROCNI[ROCNI.rok >= 2011].dropna()
m = fe("m1 ~ p1", Q)
rows.append({"model": "prevence u PL (p. b. → p. b.), 2011–2024", "proměnná": "prevence u PL",
             "elasticita": m.params["p1"], "SE (shluky)": m.bse["p1"], "p": m.pvalues["p1"], "n": int(m.nobs)})
FE = pd.DataFrame(rows)
FE.to_csv(OUT / "panel_fe.csv", index=False)
FE.round(3)

,model,proměnná,elasticita,SE (shluky),p,n
0,"gyn. prevence, 2019–2024",gyn_prevence,0.425,0.133,0.001,462
1,"gyn. prevence, bez 2020–21",gyn_prevence,0.397,0.175,0.023,308
2,gyn. + cytologie + test OK,gyn_prevence,0.310,0.117,0.008,462
3,gyn. + cytologie + test OK,cytologie,0.268,0.089,0.003,462
4,gyn. + cytologie + test OK,toks,0.032,0.043,0.459,462
5,"prevence u PL (p. b. → p. b.), 2011–2024",prevence u PL,-0.005,0.036,0.881,1078


**Jak to číst.**

* **Okresy se sbližují.** Rozdíly mezi okresy (variační koeficient ročního
  pokrytí) klesly od roku 2010 na zhruba dvě pětiny, většinou do roku 2014.
  Pořadí okresů je přitom dost stálé (Spearmanova korelace pořadí výše). Kdo byl
  pozadu, dohání, ale pořadí se mění pomalu.
* **Covid** srazil pokrytí v roce 2020 v průměru asi o 8 %, mezi okresy velmi
  nerovnoměrně. Propad ani dohnání nesouvisí výrazně s dostupností ani se
  socioekonomikou (|r| ≤ 0,25, nic průkazného po zohlednění počtu testů).
* **Uvnitř okresů jdou spolu gynekologická prevence a mamografie**: když v okrese
  přibude pacientek na preventivní gynekologické prohlídce o 1 %, přibude
  mamografií zhruba o 0,4 %. Drží to i bez covidových let. Je to silnější doklad
  „gynekologického kanálu“ než průřezová korelace, protože odečítá všechno
  trvalé, čím se okresy liší.
* **Prevence u praktika jde s mamografií nezávisle i v čase** (koeficient ≈ 0).
  Zjištění explorace se tedy potvrzuje dvakrát: mezi okresy i uvnitř nich.
* Omezení: elasticitu mohou táhnout i společné místní šoky v čerpání péče
  (např. odchod gynekologa ovlivní obojí). Na kauzální efekt „víc
  gynekologických prohlídek → víc mamografií“ to nestačí, na důkaz společného
  kanálu ano.

## 7 · Co z původních závěrů platí

Tabulka se plní z výsledků výše. Hlavní zjištění explorace (`README.md`)
vedle nového odhadu.

In [24]:
def r_(a, b):
    g = pd.concat([a, b], axis=1).dropna()
    return stats.pearsonr(g.iloc[:, 0], g.iloc[:, 1])

kr45, kr5069 = KRAJ_T.loc["hrubé pokrytí 45+ (explorace)"], KRAJ_T.loc["50–69 let"]
fit = FITY.set_index(["cíl", "model"])
c_ = STD_POPIS["m_std"]
fe_gyn = FE[(FE.model == "gyn. prevence, 2019–2024") & (FE["proměnná"] == "gyn_prevence")].iloc[0]
fe_pl = FE[FE["proměnná"] == "prevence u PL"].iloc[0]
r_pl = r_(S.p_std, S.m_std); r_gy = r_(X.gyn_prevence_pct, S.m_std)
r_sf = r_(X.sfca_30, S.m_std); r_doj = r_(X.dojizdka_opr, S.m_std)
r_ma = r_(X.ma_mamocentrum, S.m_std)
r_vek = r_(X.prumerny_vek_zeny, S.m_std)

REVIZE = pd.DataFrame([
    ("Prevence u praktika a mamografie spolu nesouvisejí (r = 0,12)",
     f"standardizovaně r = {cz(r_pl[0], 2)} (p = {cz(r_pl[1], 2)}); uvnitř okresů v čase β = {cz(fe_pl.elasticita, 3)}",
     "platí, i v čase"),
    ("Nejsilnější je gynekologická prevence (r = 0,46), dál cytologie a test OK",
     f"r = {cz(r_gy[0], 2)}; v panelu elasticita {cz(fe_gyn.elasticita, 2)} (p = {cz(fe_gyn.p, 3)}); "
     f"index prevence nejsilnější prediktor M3",
     "platí, posíleno"),
    ("Vzdálenost k pracovišti nesouvisí",
     f"kapacita v dosahu 30 km (2SFCA): r = {cz(r_sf[0], 2)} (p = {cz(r_sf[1], 2)}); "
     f"vlastní pracoviště v okrese ale ano: r = {cz(r_ma[0], 2)} (p = {cz(r_ma[1], 3)}), stabilní i v M3",
     "platí i s lepší mírou; záleží na pracovišti v okrese"),
    ("Měřená dojížďka souvisí slabě (r = −0,30), korekci těsně nepřežije",
     f"7 okresů s pracovištěm má dojížďku 100 % jen kvůli vykazování řetězců; "
     f"bez nich r = {cz(r_doj[0], 2)} (p = {cz(r_doj[1], 3)})",
     "platí, i po opravě dat; kauzálně nečíst"),
    ("Pětina rozdílů leží mezi kraji → organizace péče",
     f"45+: η² {cz(100*kr45['η²'], 0)} % vs. {cz(100*kr45['η² náhodou'], 0)} % náhodou, p = {cz(kr45['p (permutace)'], 2)}; "
     f"50–69 let: η² {cz(100*kr5069['η²'], 0)} %, p = {cz(kr5069['p (permutace)'], 3)}, ICC {cz(kr5069['ICC'], 2)}",
     "pro 45+ neprůkazné, pro 50–69 platí"),
    ("Průměrný věk žen je nejsilnější prediktor modelu (β = −2,4)",
     f"sám o sobě r = {cz(r_vek[0], 2)}; ve starém modelu VIF {cz(VIF_OLD['prumerny_vek_zeny'], 1)}",
     "artefakt přeučeného modelu"),
    ("Model s 22 prediktory: adj. R² 0,52",
     f"R² na datech {cz(fit.loc[(c_, 'model explorace (22)'), 'R²'], 2)}, mimo vzorek "
     f"{cz(fit.loc[(c_, 'model explorace (22)'), 'R² 5-fold CV'], 2)}; "
     f"M3 (9 prediktorů) mimo vzorek {cz(fit.loc[(c_, 'M3 + prevence'), 'R² 5-fold CV'], 2)}",
     "přeučený; M3 stejně dobrý s 9 prediktory"),
    ("Socioekonomika pokrytí jednotlivě nevysvětluje",
     f"ani dohromady: R² mimo vzorek {cz(fit.loc[(c_, 'M1 socioekonomika'), 'R² 5-fold CV'], 2)}",
     "platí"),
    ("(nové) Okresy jsou nezávislá pozorování",
     f"Moran I = {cz(MORAN.loc['standardizované pokrytí 45+', 'Moran I'], 2)} "
     f"(p = {cz(MORAN.loc['standardizované pokrytí 45+', 'p'], 3)})",
     "neplatí, p hodnoty explorace jsou optimistické"),
    ("(nové) Vývoj v čase",
     f"variační koeficient mezi okresy {cz(100*SIG.cv.iloc[0], 0)} % (2010) → {cz(100*SIG.cv.iloc[-1], 0)} % ({SIG.index[-1]})",
     "okresy se sbližují"),
], columns=["původní závěr", "nový odhad", "stav"])
REVIZE.to_csv(OUT / "revize_zjisteni.csv", index=False)
with pd.option_context("display.max_colwidth", 200):
    display(REVIZE.style.hide(axis="index").set_properties(**{"text-align": "left"}))

původní závěr,nový odhad,stav
"Prevence u praktika a mamografie spolu nesouvisejí (r = 0,12)","standardizovaně r = 0,08 (p = 0,52); uvnitř okresů v čase β = -0,005","platí, i v čase"
"Nejsilnější je gynekologická prevence (r = 0,46), dál cytologie a test OK","r = 0,46; v panelu elasticita 0,42 (p = 0,001); index prevence nejsilnější prediktor M3","platí, posíleno"
Vzdálenost k pracovišti nesouvisí,"kapacita v dosahu 30 km (2SFCA): r = 0,02 (p = 0,83); vlastní pracoviště v okrese ale ano: r = 0,23 (p = 0,041), stabilní i v M3",platí i s lepší mírou; záleží na pracovišti v okrese
"Měřená dojížďka souvisí slabě (r = −0,30), korekci těsně nepřežije","7 okresů s pracovištěm má dojížďku 100 % jen kvůli vykazování řetězců; bez nich r = -0,33 (p = 0,006)","platí, i po opravě dat; kauzálně nečíst"
Pětina rozdílů leží mezi kraji → organizace péče,"45+: η² 22 % vs. 17 % náhodou, p = 0,18; 50–69 let: η² 34 %, p = 0,006, ICC 0,22","pro 45+ neprůkazné, pro 50–69 platí"
"Průměrný věk žen je nejsilnější prediktor modelu (β = −2,4)","sám o sobě r = 0,02; ve starém modelu VIF 4,6",artefakt přeučeného modelu
"Model s 22 prediktory: adj. R² 0,52","R² na datech 0,68, mimo vzorek 0,24; M3 (9 prediktorů) mimo vzorek 0,29",přeučený; M3 stejně dobrý s 9 prediktory
Socioekonomika pokrytí jednotlivě nevysvětluje,"ani dohromady: R² mimo vzorek 0,07",platí
(nové) Okresy jsou nezávislá pozorování,"Moran I = 0,20 (p = 0,004)","neplatí, p hodnoty explorace jsou optimistické"
(nové) Vývoj v čase,variační koeficient mezi okresy 21 % (2010) → 8 % (2024),okresy se sbližují


## 8 · Metodické poznámky

1. **Standardizace** používá jako standard věkovou skladbu žen 45+ celé ČR
   (2020–2024). Pásmo 85+ je otevřené, takže drobné rozdíly ve skladbě uvnitř
   něj standardizace nezachytí.
2. **2SFCA** počítá s těžišti okresů, ne s bydlištěm žen. Uvnitř velkých
   okresů to dostupnost zprůměruje. Vzdálenosti jsou vzdušné.
3. **Zkreslení okresu zařízení v NRHZS** (kap. 3.1) se týká všech ukazatelů
   počítaných podle okresu poskytovatele. Ukazatele podle okresu *pacienta*
   (gynekologie, cytologie, test OK, mamografie v panelu) zasažené nejsou.
4. **Sousedství** se určuje z generalizovaných hranic s tolerancí ~400 m, takže
   může vynechat okresy, které se dotýkají jen v bodě.
5. **Smíšený model** se 14 kraji odhaduje rozptyl kraje nepřesně. Proto
   hlavní test role krajů stojí na permutaci (kap. 2).
6. Všechny vztahy zůstávají **ekologické**. Z chování okresů nelze usuzovat na
   chování jednotlivých žen.